<a href="https://colab.research.google.com/github/MoAppOfficial/moapp-colab/blob/main/%D8%AA%D8%AD%D9%85%D9%8A%D9%84_%D9%85%D8%AC%D9%84%D8%AF%D8%A7%D8%AA_%D8%AC%D9%88%D9%81%D8%A7%D9%8A%D9%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

#@title 🚀 لوحة تحميل جوفايل التفاعلية
#@markdown ---
#@markdown ### 🛠️ 1. إعدادات التثبيت
#@markdown حدد هذا المربع إذا كانت هذه أول مرة تشغل فيها الكود في هذه الجلسة لتثبيت الحزم المطلوبة:
تثبيت_الحزم = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🔗 2. رابط المجلد
رابط_المجلد = "" #@param {type:"string", placeholder:"ضع رابط Gofile هنا"}

#@markdown ---
#@markdown ### 💾 3. طريقة الحفظ
طريقة_الحفظ = "\u0627\u0644\u062A\u062D\u0645\u064A\u0644 \u0627\u0644\u0645\u0628\u0627\u0634\u0631 \u0645\u0646 \u062C\u0648\u0641\u0627\u064A\u0644 \u0643\u0645\u0644\u0641 \u0645\u0636\u063A\u0648\u0637 (\u064A\u0646\u0635\u062D \u0628\u0647\u0627 \u0644\u0640 \"\u0645\u0644\u0641\u0627\u062A \u0627\u0643\u062B\u0631 \u0645\u0646 1 \u062C\u064A\u062C\u0627\")" #@param ["النسخ إلى حساب بجوجل درايف", "التحميل المباشر من جوفايل كملف مضغوط (ينصح بها لـ \"ملفات اكثر من 1 جيجا\")", "التحميل المباشر من المتصفح كملف مضغوط (لا ينصح بها لـ \"ملفات اكثر من 1 جيجا\")"]

import os
import time
import json
import shutil
import requests
from IPython.display import clear_output

# ---------------------------------------------------------
# 1. تثبيت الحزم
# ---------------------------------------------------------
if تثبيت_الحزم:
    print("⏳ جاري تثبيت الحزم المطلوبة... يرجى الانتظار.")

    get_ipython().system(
        'wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb > /dev/null 2>&1'
    )

    get_ipython().system(
        'apt-get install -y ./google-chrome-stable_current_amd64.deb -qq > /dev/null 2>&1'
    )

    get_ipython().system(
        'pip install -q selenium yt-dlp requests > /dev/null 2>&1'
    )

    clear_output()
    print("✅ تم تثبيت الحزم بنجاح!\n")

import yt_dlp
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# ---------------------------------------------------------
# دالة رفع ملف الـ ZIP إلى Gofile
# ---------------------------------------------------------
def upload_to_gofile(file_path, token=None):
    print("🌐 جاري فحص روابط الرفع من جوفايل...")
    try:
        server_res = requests.get("https://api.gofile.io/servers").json()
        if server_res.get("status") != "ok":
            print("❌ تعذر جلب سيرفرات الرفع من جوفايل.")
            return None

        server = server_res["data"]["servers"][0]["name"]
        upload_url = f"https://{server}.gofile.io/contents/uploadfile"

        print(f"🚀 جاري رفع الملف المضغوط إلى جوفايل مجدداً...")
        with open(file_path, "rb") as f:
            data = {"token": token} if token else {}
            response = requests.post(upload_url, files={"file": f}, data=data)

        res_json = response.json()
        if res_json.get("status") == "ok":
            return res_json["data"]
        else:
            print(f"⚠️ خطأ أثناء الرفع: {res_json}")
            return None
    except Exception as e:
        print(f"⚠️ فشل الرفع: {e}")
        return None

# ---------------------------------------------------------
# التشغيل الرئيسي
# ---------------------------------------------------------
def main():
    if not رابط_المجلد or "gofile.io" not in رابط_المجلد:
        print("❌ خطأ: يرجى وضع رابط Gofile صحيح.")
        return

    folder_id = رابط_المجلد.rstrip('/').split('/')[-1]

    print("🌐 جاري تجهيز الحزم المطلوبة...")
    chrome_options = Options()
    chrome_options.add_argument('--headless=new')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.set_capability('goog:loggingPrefs', {'performance': 'ALL'})

    driver = webdriver.Chrome(options=chrome_options)
    print("🔗 جاري فتح الرابط...")
    driver.get(رابط_المجلد)
    time.sleep(10)

    # سحب التوكن
    token = driver.execute_script("return localStorage.getItem('accountToken');")
    if not token:
        for cookie in driver.get_cookies():
            if cookie['name'] == 'accountToken':
                token = cookie['value']
                break

    # اعتراض ملف الـ JSON
    folder_data = None
    logs = driver.get_log('performance')
    for entry in logs:
        try:
            log = json.loads(entry['message'])['message']
            if log['method'] == 'Network.responseReceived':
                url = log['params']['response']['url']
                if 'api.gofile.io/contents/' in url:
                    req_id = log['params']['requestId']
                    body = driver.execute_cdp_cmd('Network.getResponseBody', {'requestId': req_id})
                    folder_data = json.loads(body['body'])
                    break
        except Exception:
            continue

    driver.quit()

    if not folder_data or folder_data.get('status') != 'ok':
        print("⚠️ فشل في جلب بيانات المجلد.")
        return

    files_info = []
    children = folder_data.get('data', {}).get('children', {})
    for key, item in children.items():
        if item.get('type') == 'file':
            files_info.append({'name': item.get('name'), 'link': item.get('link')})

    if not files_info:
        print("⚠️ لم يتم العثور على ملفات داخل هذا المجلد.")
        return

    # عرض الملفات واختيارها
    print("\n" + "="*50)
    print("📁 قائمة الملفات المتاحة للتحميل:")
    print("="*50)
    for i, f in enumerate(files_info):
        print(f"[{i}] {f['name']}")
    print("="*50)

    choice = input("\n✍️ أدخل أرقام الملفات مفصولة بفاصلة (مثال: 0,2) أو اتركه فارغاً لتحميل الكل: ").strip()
    selected_files = []
    if choice:
        try:
            indices = [int(x.strip()) for x in choice.split(',') if x.strip().isdigit()]
            for idx in indices:
                if 0 <= idx < len(files_info):
                    selected_files.append(files_info[idx])
            print(f"\n✅ تم اختيار {len(selected_files)} ملف/ات.\n")
        except Exception:
            print("\n⚠️ خطأ في الإدخال، سيتم تحميل جميع الملفات.\n")
            selected_files = files_info
    else:
        print("\n✅ سيتم تحميل جميع الملفات.\n")
        selected_files = files_info

    if not selected_files:
        return

    base_folder_name = selected_files[0]['name'].rsplit('.', 1)[0]

    if طريقة_الحفظ == "النسخ إلى درايف":
        from google.colab import drive
        drive.mount('/content/drive')
        DOWNLOAD_DIR = f"/content/drive/My Drive/GoFile/{base_folder_name}"
        os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    else:
        DOWNLOAD_DIR = f"/content/gofile_temp/{base_folder_name}"
        if os.path.exists(DOWNLOAD_DIR):
            shutil.rmtree(DOWNLOAD_DIR)
        os.makedirs(DOWNLOAD_DIR, exist_ok=True)

    # التحميل الفعلي
    print("\n🚀 بدء عملية التحميل إلى كولاب...")
    dl_headers = {
        'Cookie': f'accountToken={token}',
        'Referer': 'https://gofile.io/',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    download_opts = {
        'outtmpl': f'{DOWNLOAD_DIR}/%(title)s.%(ext)s',
        'http_headers': dl_headers,
        'continuedl': True,
        'ignoreerrors': True,
        'nocheckcertificate': True
    }

    with yt_dlp.YoutubeDL(download_opts) as ydl:
        ydl.download([f['link'] for f in selected_files])

    print("\n✅ اكتمل تجهيز الملفات!")

        # ---------------------------------------------------------
    # الإجراء بناءً على طريقة الحفظ المختارة
    # ---------------------------------------------------------
    if "جوجل درايف" in طريقة_الحفظ:
        print(f"🎉 تم حفظ الملفات في جوجل درايف: {DOWNLOAD_DIR}")

    elif "جوفايل" in طريقة_الحفظ:
        print("\n🗜️ جاري ضغط الملفات في ملف واحد...")
        zip_base = f"/content/{base_folder_name}"
        shutil.make_archive(zip_base, 'zip', DOWNLOAD_DIR)
        final_zip_path = f"{zip_base}.zip"

        size_mb = os.path.getsize(final_zip_path) / (1024 * 1024)
        print(f"📦 حجم الملف المضغوط: {size_mb:.2f} ميجابايت")

        # رفع الـ ZIP إلى Gofile
        upload_data = upload_to_gofile(final_zip_path, token=token)
        if upload_data:
            print("\n" + "🎉"*20)
            print("✅ تم تجهيز ورفع الملف بنجاح!")
            print(f"🔗 رابط التحميل: {upload_data.get('downloadPage')}")
            print("🎉"*20)
            print("💡 افتح الرابط في متصفحك وقم بتحميل الملف المضغوط")

    elif "المتصفح" in طريقة_الحفظ:
        from google.colab import files
        print("\n🗜️ جاري ضغط الملفات...")
        zip_base = f"/content/{base_folder_name}"
        shutil.make_archive(zip_base, 'zip', DOWNLOAD_DIR)
        print("✅ تم الضغط بنجاح جارى التحميل بالخلفية (لا تغلق علامة التبويب)")
        files.download(f"{zip_base}.zip")

main()

✅ تم تثبيت الحزم بنجاح!

🌐 جاري تجهيز الحزم المطلوبة...
🔗 جاري فتح الرابط...

📁 قائمة الملفات المتاحة للتحميل:
[0] Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR.mkv
[1] Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR.sub.EN.vtt

✍️ أدخل أرقام الملفات مفصولة بفاصلة (مثال: 0,2) أو اتركه فارغاً لتحميل الكل: 


Deprecated Feature: Passing cookies as a header is a potential security risk; they will be scoped to the domain of the downloaded urls. Please consider loading cookies from a file or browser instead.



✅ سيتم تحميل جميع الملفات.


🚀 بدء عملية التحميل إلى كولاب...
[generic] Extracting URL: https://store10.gofile.io/download/web/f1543ac9-1c20-4e47-b074-4835ff4d5683/Chaharshanbeh.19.Ordi...C.2.0.H.264-M3TR.mkv
[generic] Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR: Downloading webpage
[info] Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR: Downloading 1 format(s): x-matroska
[download] Destination: /content/gofile_temp/Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR/Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR.mkv
[download] 100% of    2.28GiB in 00:00:50 at 46.58MiB/s  
[generic] Extracting URL: https://store1.gofile.io/download/web/b2663bbe-0737-47f8-8354-5ab04b126939/Chaharshanbeh.19.Ordib....264-M3TR.sub.EN.vtt
[generic] Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR.sub.EN: Downloading webpage


[info] Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR.sub.EN: Downloading 1 format(s): 0
[download] Destination: /content/gofile_temp/Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR/Chaharshanbeh.19.Ordibehesht.2015.1080p.WEB-DL.AAC.2.0.H.264-M3TR.sub.EN.vtt
[download] 100% of   97.96KiB in 00:00:00 at 520.17KiB/s 

✅ اكتمل تجهيز الملفات!

🗜️ جاري ضغط الملفات في ملف واحد...
📦 حجم الملف المضغوط: 2331.79 ميجابايت
🌐 جاري فحص روابط الرفع من جوفايل...
🚀 جاري رفع الملف المضغوط إلى جوفايل مجدداً...

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
✅ تم تجهيز ورفع الملف بنجاح!
🔗 رابط التحميل: https://gofile.io/d/ItfVbSIZ
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
💡 افتح الرابط في متصفحك وقم بتحميل الملف المضغوط
